# Semana 10: Aprendizaje no supervisado

## Perfiles de impacto ambiental de la energia

Este notebook usa la tabla analitica generada en Semana 9 para agrupar combinaciones region-categoria energetica segun sus niveles de generacion y emisiones. Se aplican PCA, K-means y DBSCAN siguiendo la pauta.

## Variables del modelo

- `log_generacion`: escala logaritmica del volumen generado.
- `log_emisiones`: escala logaritmica de emisiones CO2.
- `intensidad_co2`: toneladas CO2 por MWh.
- `participacion_generacion`: proporcion de la generacion de la region correspondiente a la categoria.

Las variables son estandarizadas antes de PCA y K-means para que sus escalas originales no dominen las distancias.

In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import display, Image

OUTPUT_DIR = Path('salidas')
INPUT_PATH = Path('../Semana 9 EDA Spark/salidas/features_eda_semana9.csv')
assert INPUT_PATH.exists(), 'Ejecuta primero EDA_Semana9.ipynb para crear las caracteristicas.'
print('Entrada:', INPUT_PATH)

## PCA, seleccion de k y clustering

El script compara valores de `k` entre 2 y 10 con inercia y silhouette. En lugar de fijar arbitrariamente `k=3`, selecciona el mayor silhouette y reporta los perfiles obtenidos.

Ademas, guarda los registros con su cluster K-means en `modelos/datos_etiquetados_kmeans` (Parquet) y el modelo en `modelos/kmeans_energia_v1`. Estas pseudo-etiquetas son la entrada de Semana 12.

In [ ]:
%run ./clustering_semana10.py

In [ ]:
evaluacion = pd.read_csv(OUTPUT_DIR / 'evaluacion_kmeans.csv')
clusters_kmeans = pd.read_csv(OUTPUT_DIR / 'clusters_kmeans.csv')
clusters_dbscan = pd.read_csv(OUTPUT_DIR / 'clusters_dbscan.csv')
display(evaluacion)
display(clusters_kmeans.head(10))

## K-means: visualizacion PCA

PCA reduce las cuatro variables estandarizadas a dos componentes para observar los grupos. La segmentacion facilita reconocer perfiles de alto volumen, mayor intensidad o baja participacion regional.

In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'figuras' / 'seleccion_k.png')))
display(Image(filename=str(OUTPUT_DIR / 'figuras' / 'clusters_kmeans_pca.png')))

## DBSCAN: deteccion de densidad y ruido

DBSCAN se ejecuta sobre las coordenadas PCA. El cluster `-1` identifica observaciones aisladas que conviene examinar como perfiles atipicos y no descartar automaticamente. Sus parametros pueden ajustarse con `DBSCAN_EPS` y `DBSCAN_MIN_SAMPLES`.

In [ ]:
display(clusters_dbscan['cluster_dbscan'].value_counts().sort_index().rename('observaciones').to_frame())
display(Image(filename=str(OUTPUT_DIR / 'figuras' / 'clusters_dbscan_pca.png')))

## Lectura para el proyecto

- K-means entrega segmentos comparables para caracterizar patrones energeticos frecuentes.
- DBSCAN complementa el analisis detectando observaciones que no pertenecen a concentraciones densas.
- Un perfil de elevada intensidad CO2 es una senal descriptiva para priorizar analisis, no una recomendacion causal por si sola.
- La interpretacion debe mantener visible el periodo, region y categoria energetica de cada punto.